# OCEANIQ — end-to-end pipeline (pillars 1 -> 2)

**First real integration.** Everything so far has been proven in isolation. This
runs one scene through the actual chain:

```
SAR image -> U-Net mask -> look-alike screening -> lat/lon seeds
          -> OpenDrift BACKWARD -> origin bbox + time window -> fixtures/*.json
```

The JSON it writes is the Pillar 1->2 and 2->3 payload defined in
`contracts/georeferencing.json`, and it unblocks Agent B's scoring engine.

---

## Two honest placeholders — read before quoting any result

**1. Geography is a placeholder.** The only forcing data available today is
OpenDrift's bundled **NorKyst sample (western Norway)**. The demo transform would
place the spill in the Arabian Sea, where we have no currents — particles would
not move at all. So the anchor is set **inside the NorKyst domain** (~4.9E,
60.0N) to prove the plumbing.

Swapping in real geography is a **config change, not a rewrite**: obtain CMEMS
currents for Indian waters, set `ANCHOR_LON/ANCHOR_LAT` to the true scene corner,
point the reader at the CMEMS file.

**2. Georeferencing is Path B.** Assumed anchor and pixel size, not a real
geotransform — the Deep-SAR PNGs carry none. **State this on the slide.**

Setup: GPU runtime, and Kaggle secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`) with
Notebook access **ON for this notebook** — that toggle is per-notebook.

## 1. Install

In [ ]:
!pip install -q "segmentation-models-pytorch>=0.3.4" "albumentations>=1.4,<2.0" opendrift

import torch, opendrift
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("opendrift", opendrift.__version__)

## 2. Mount Drive (auto-retrying)

In [ ]:
from google.colab import drive
import os, time

def mount_drive(attempts=4, wait=6):
    """drive.mount() fails intermittently with a bare ValueError('mount failed');
    it succeeds on a plain retry. Retry rather than stall a live demo."""
    for i in range(1, attempts + 1):
        if os.path.isdir("/content/drive/MyDrive"):
            print("Drive already mounted"); return
        try:
            drive.mount("/content/drive", force_remount=(i > 1))
            print(f"Drive mounted (attempt {i})"); return
        except Exception as e:
            print(f"  attempt {i}/{attempts}: {type(e).__name__}: {e}")
            if i < attempts: time.sleep(wait)
    raise RuntimeError("Drive mount failed - re-run this cell")

mount_drive()

## 3. Config

In [ ]:
from pathlib import Path

CKPT_PATH = Path("/content/drive/MyDrive/oil_spill_runs/unet_resnet34_best.pth")
KAGGLE_DATASET = "bakhtiyar2222/deep-sar-oil-spill-segmentation-refined"
RAW_DIR, DATA_ROOT = Path("/content/raw"), Path("/content/oil_spill")
FIXTURES = Path("/content/fixtures"); FIXTURES.mkdir(exist_ok=True)

# --- PLACEHOLDER GEOGRAPHY: inside the NorKyst sample domain so drift works ---
ANCHOR_LON, ANCHOR_LAT = 4.85, 60.05
PIXEL_SIZE_DEG = 0.0001          # ~11 m/px, Sentinel-1 GRD order of magnitude

# For real Indian waters (needs CMEMS currents first):
# ANCHOR_LON, ANCHOR_LAT = 69.10, 18.52

DRIFT_HOURS = 12
MAX_SEEDS = 300
print("anchor:", ANCHOR_LON, ANCHOR_LAT, "| PLACEHOLDER geography")

## 4. Get the data and the trained model

In [ ]:
import os, shutil
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

if not (DATA_ROOT / "images" / "val").is_dir():
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    print("downloading dataset...")
    api.dataset_download_files(KAGGLE_DATASET, path=str(RAW_DIR), unzip=True, quiet=False)
    for kind in ("images", "masks"):
        for split in ("train", "val"):
            src = RAW_DIR / kind / kind / split
            dst = DATA_ROOT / kind / split
            dst.parent.mkdir(parents=True, exist_ok=True)
            if src.is_dir() and not dst.exists():
                shutil.move(str(src), str(dst))
print("val images:", len(list((DATA_ROOT / "images" / "val").iterdir())))

assert CKPT_PATH.exists(), f"checkpoint not found: {CKPT_PATH}"
print("checkpoint:", CKPT_PATH, f"({CKPT_PATH.stat().st_size/1e6:.0f} MB)")

## 5. Look-alike screening module (embedded from `lookalike_screen.py`)

In [ ]:
"""Heuristic look-alike screening for binary oil-spill segmentation masks.

NOT a trained classifier. These are hand-written shape rules applied to the
blobs a segmentation model already produced. They encode one domain
observation: wind- and current-driven oil slicks tend to be elongated with
ragged edges, while common SAR look-alikes (algal blooms, calm-water "black
holes", biogenic films) tend to be small, round and smooth.

The rules are a screening aid, not evidence. A real look-alike classifier needs
labelled look-alike examples -- the restricted-access Krestenitis/MKLab 5-class
dataset (sea / oil spill / look-alike / ship / land, mklab.iti.gr, by request).
That is future work.

Depends only on numpy + scipy.ndimage, both preinstalled in Colab.
"""

import numpy as np
from scipy import ndimage

LABEL = "heuristic look-alike screening (rule-based, not a trained classifier)"

# --- thresholds -------------------------------------------------------------
# Calibrated against synthetic discs/ellipses; see demo() at the bottom.
# Tune these on your own imagery -- they are the knobs, not the algorithm.
MIN_AREA_PX = 50            # below this, treat as speckle noise, not a detection
SMALL_AREA_PX = 600         # "small" for the round-and-smooth rule
ROUND_ELONGATION_MAX = 1.8  # <= this is "not elongated" (1.0 = circle)
SMOOTH_ROUGHNESS_MAX = 1.35 # <= this is "smooth-edged" (1.0 = perfect circle)


def _perimeter_px(blob):
    """Boundary-pixel count: blob pixels having at least one 4-neighbour outside.

    A digital approximation, and it reads LOW on smooth curves: a disc measures
    roughness ~0.86, not the textbook 1.0, because one boundary pixel can cover
    more than one unit of arc. Thresholds below are set from measured shapes for
    exactly this reason -- do not re-derive them from circularity theory.
    """
    p = np.pad(blob, 1)
    interior = p[:-2, 1:-1] & p[2:, 1:-1] & p[1:-1, :-2] & p[1:-1, 2:]
    return int((blob & ~interior).sum())


def _elongation(ys, xs):
    """Major/minor axis ratio from second moments. 1.0 = round, higher = longer."""
    if len(ys) < 2:
        return 1.0
    cov = np.cov(np.stack([ys.astype(float), xs.astype(float)]))
    cov = np.atleast_2d(cov)
    if not np.all(np.isfinite(cov)):
        return 1.0
    eigs = np.linalg.eigvalsh(cov)
    lo, hi = float(max(eigs.min(), 0.0)), float(max(eigs.max(), 0.0))
    if hi <= 0:
        return 1.0
    if lo <= 1e-9:
        return float("inf")     # perfectly straight line: maximally elongated
    return float(np.sqrt(hi / lo))


def blob_features(mask):
    """One dict per connected blob: area, elongation, roughness.

    roughness = perimeter / perimeter of a circle of the same area.
    1.0 is a smooth disc; larger means a more ragged outline.
    """
    binary = np.asarray(mask) > 0
    labels, n = ndimage.label(binary)
    out = []
    for i in range(1, n + 1):
        blob = labels == i
        area = int(blob.sum())
        ys, xs = np.nonzero(blob)
        perim = _perimeter_px(blob)
        equiv = 2.0 * np.sqrt(np.pi * area)          # circle of equal area
        out.append({
            "label": i,
            "area_px": area,
            "elongation": round(_elongation(ys, xs), 3),
            "roughness": round(perim / equiv, 3) if equiv > 0 else 0.0,
            "perimeter_px": perim,
            "centroid_yx": (round(float(ys.mean()), 1), round(float(xs.mean()), 1)),
        })
    return labels, out


def classify(f,
             min_area_px=MIN_AREA_PX,
             small_area_px=SMALL_AREA_PX,
             round_elongation_max=ROUND_ELONGATION_MAX,
             smooth_roughness_max=SMOOTH_ROUGHNESS_MAX):
    """Three rules, in order. Returns (verdict, reason)."""
    if f["area_px"] < min_area_px:
        return "noise", f"area {f['area_px']}px < {min_area_px}px"

    if (f["elongation"] <= round_elongation_max
            and f["roughness"] <= smooth_roughness_max
            and f["area_px"] < small_area_px):
        return "look-alike", (
            f"round (elong {f['elongation']:.2f}), smooth (rough "
            f"{f['roughness']:.2f}), small ({f['area_px']}px)"
        )

    return "oil", (f"elong {f['elongation']:.2f}, rough {f['roughness']:.2f}, "
                   f"{f['area_px']}px")


def screen(mask, **thresholds):
    """Filter a binary prediction mask.

    Returns (kept_mask, blobs); kept_mask keeps only 'oil' blobs, and each blob
    dict carries 'verdict' and 'reason'.
    """
    labels, blobs = blob_features(mask)
    kept = np.zeros(labels.shape, dtype=np.uint8)
    for f in blobs:
        f["verdict"], f["reason"] = classify(f, **thresholds)
        if f["verdict"] == "oil":
            kept[labels == f["label"]] = 1
    return kept, blobs


def report(blobs, max_rows=20):
    """Plain-text summary. Always names the method, so downstream output cannot
    accidentally present this as a trained classifier's result."""
    header = LABEL.upper()
    if not blobs:
        return f"{header}\n0 blobs detected"
    counts = {}
    for f in blobs:
        counts[f["verdict"]] = counts.get(f["verdict"], 0) + 1
    lines = [
        header,
        f"{len(blobs)} blobs: " + ", ".join(f"{v} {k}" for k, v in sorted(counts.items())),
        f"{'id':>3} {'area':>7} {'elong':>6} {'rough':>6}  verdict",
    ]
    for f in sorted(blobs, key=lambda b: -b["area_px"])[:max_rows]:
        lines.append(f"{f['label']:>3} {f['area_px']:>7} {f['elongation']:>6.2f} "
                     f"{f['roughness']:>6.2f}  {f['verdict']}")
    if len(blobs) > max_rows:
        lines.append(f"... {len(blobs) - max_rows} more")
    return "\n".join(lines)

## 6. Pixel to lat/lon (embedded from `tools/spill_to_seeds.py`)

In [ ]:
import numpy as np

def mask_to_seed_points(mask, transform, when):
    """
    Converts a binary pixel mask of a spill into a list of real-world seed points (lat, lon, time).
    
    Args:
        mask (np.ndarray): 256x256 binary mask (e.g., >127 is oil).
        transform (tuple or object): Geotransform to convert (x, y) pixels to (lon, lat).
                                     For the demo, we pass an anchoring function.
        when (str or datetime): The timestamp of the satellite image (when the spill was discovered).
        
    Returns:
        list of dict: [{"lat": lat, "lon": lon, "time": when}, ...]
    """
    points = []
    
    # 1. Find all pixel coordinates where the mask is positive (oil)
    # mask > 127 is the threshold validated in Pillar 1
    y_coords, x_coords = np.where(mask > 127)
    
    # 2. To avoid simulating millions of particles for large spills,
    # we can sample the points or just take every Nth pixel.
    # For demo purposes, let's take up to 1000 points.
    step = max(1, len(x_coords) // 1000)
    
    for x, y in zip(x_coords[::step], y_coords[::step]):
        # Apply the geotransform
        if callable(transform):
            lon, lat = transform(x, y)
        else:
            # Standard affine transform (e.g. from rasterio: ~affine.Affine)
            lon, lat = transform * (x, y)
            
        points.append({
            "lat": float(lat),
            "lon": float(lon),
            "time": when
        })
        
    return points

def get_demo_transform(anchor_lon=68.5, anchor_lat=18.5, pixel_size_deg=0.0001):
    """
    PATH (B) - Demo Placement
    Creates a callable transform that anchors the top-left of the 256x256 tile
    to a specific coordinate in the Arabian Sea, assuming a fixed pixel size.
    
    This is explicitly for the hackathon demo. For production (Path A), 
    Sentinel-1 GeoTIFFs should be used with rasterio.transform.
    """
    def transform(x, y):
        # lon increases to the right (x), lat decreases downwards (y)
        lon = anchor_lon + (x * pixel_size_deg)
        lat = anchor_lat - (y * pixel_size_deg)
        return lon, lat
    return transform

## 7. STEP 1 — detect: run the trained model on a real SAR image

In [ ]:
import numpy as np, torch
from PIL import Image
import segmentation_models_pytorch as smp

ENCODER, NUM_CLASSES = "resnet34", 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
params = smp.encoders.get_preprocessing_params(ENCODER, "imagenet")
MEAN, STD = np.array(params["mean"]), np.array(params["std"])

model = smp.Unet(ENCODER, encoder_weights=None, in_channels=3,
                 classes=NUM_CLASSES, activation=None).to(DEVICE)
ck = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ck["model"]); model.eval()
print(f"loaded checkpoint: epoch {ck['epoch']} | mIoU {ck['miou']:.4f}")

val_imgs = sorted((DATA_ROOT / "images" / "val").iterdir())
chosen, pred = None, None
for p in val_imgs[:40]:                      # find a scene with real oil in it
    img = np.array(Image.open(p).convert("RGB"))
    x = ((img / 255.0 - MEAN) / STD).transpose(2, 0, 1)[None].astype("float32")
    with torch.no_grad():
        logits = model(torch.from_numpy(x).to(DEVICE))
    m = (logits.argmax(1)[0].cpu().numpy() == 1).astype(np.uint8)
    if 0.02 < m.mean() < 0.45:
        chosen, pred = p, m
        break

assert chosen is not None, "no suitable validation scene found"
print(f"scene: {chosen.name}")
print(f"predicted oil pixels: {int(pred.sum())} ({100*pred.mean():.2f}% of tile)")

## 8. STEP 2 — screen out look-alikes

In [ ]:
kept, blobs = screen(pred * 255)
print(report(blobs))
print()
before, after = int(pred.sum()), int(kept.sum())
print(f"pixels before screening: {before}")
print(f"pixels after screening : {after}")
print(f"removed by screening   : {before-after} ({100*(before-after)/max(1,before):.1f}%)")
assert after > 0, "screening removed everything - loosen thresholds"

## 9. STEP 3 — georeference: mask to lat/lon seed points (Pillar 1 -> 2 contract)

In [ ]:
import json, glob, urllib.request
from datetime import timezone
from opendrift.readers import reader_netCDF_CF_generic
import opendrift

NC_NAME, SUBDIR = "norkyst800_subset_16Nov2015.nc", "16Nov2015_NorKyst_z_surface"
def find_sample():
    roots = []
    tdf = getattr(opendrift, "test_data_folder", None)
    if isinstance(tdf, str): roots.append(tdf)
    pkg = os.path.dirname(opendrift.__file__)
    roots += [os.path.join(pkg, "..", "tests", "test_data"), "/content/opendrift_test_data"]
    for r in roots:
        hits = glob.glob(os.path.join(r, "**", NC_NAME), recursive=True)
        if hits: return hits[0]
    return None

nc = find_sample()
if nc is None:
    d = os.path.join("/content/opendrift_test_data", SUBDIR); os.makedirs(d, exist_ok=True)
    nc = os.path.join(d, NC_NAME)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/OpenDrift/opendrift/master/"
        f"tests/test_data/{SUBDIR}/{NC_NAME}", nc)
reader = reader_netCDF_CF_generic.Reader(nc)

OBSERVED_TIME = reader.end_time
transform = get_demo_transform(ANCHOR_LON, ANCHOR_LAT, PIXEL_SIZE_DEG)
seeds = mask_to_seed_points(kept * 255, transform,
                            OBSERVED_TIME.replace(tzinfo=timezone.utc).isoformat())[:MAX_SEEDS]

lons = [s["lon"] for s in seeds]; lats = [s["lat"] for s in seeds]
print(f"seed points: {len(seeds)}")
print(f"  lon {min(lons):.4f} -> {max(lons):.4f}")
print(f"  lat {min(lats):.4f} -> {max(lats):.4f}")
print(f"  observed at {OBSERVED_TIME}")

payload = {"seed_points": seeds, "crs": "EPSG:4326",
           "timestamp": seeds[0]["time"], "source_scene": chosen.name,
           "notes": "PLACEHOLDER geography: demo anchor inside the NorKyst sample "
                    "domain. Path B georeferencing (assumed anchor + pixel size)."}
(FIXTURES / "spill_seeds.json").write_text(json.dumps(payload, indent=2))
print("wrote fixtures/spill_seeds.json")

## 10. STEP 4 — backward drift from the real detected seeds

Not a synthetic point release: these positions came from the model's own
prediction, screened, then georeferenced.

In [ ]:
from datetime import timedelta
from opendrift.models.oceandrift import OceanDrift

o = OceanDrift(loglevel=30)
o.add_reader(reader)
o.seed_elements(lon=np.array(lons), lat=np.array(lats), time=OBSERVED_TIME)
print(f"seeded {len(lons)} particles from the detected slick")
o.run(duration=timedelta(hours=DRIFT_HOURS), time_step=-3600, time_step_output=3600)

def get_track(sim):
    res = getattr(sim, "result", None)
    if res is not None:
        try: return np.asarray(res["lon"]), np.asarray(res["lat"])
        except Exception: pass
    h = sim.history
    return np.ma.filled(h["lon"], np.nan), np.ma.filled(h["lat"], np.nan)

dlon, dlat = get_track(o)
print(f"ran backward: {o.time < o.start_time}  ({o.start_time} -> {o.time})")
assert o.time < o.start_time, "did not run backward"

## 11. STEP 5 — origin bbox + time window (Pillar 2 -> 3 contract)

In [ ]:
final_lon, final_lat = dlon[:, -1], dlat[:, -1]
ok = np.isfinite(final_lon) & np.isfinite(final_lat)
final_lon, final_lat = final_lon[ok], final_lat[ok]
print(f"particles surviving to origin: {int(ok.sum())}/{len(ok)}")

PAD = 0.02   # ~2 km - crude stand-in for a real ensemble spread
bbox = [float(final_lon.min()-PAD), float(final_lat.min()-PAD),
        float(final_lon.max()+PAD), float(final_lat.max()+PAD)]
origin_time = o.time.replace(tzinfo=timezone.utc)
observed_time = o.start_time.replace(tzinfo=timezone.utc)

out = {"origin_bbox": [round(v, 5) for v in bbox],
       "time_window": {"start": origin_time.isoformat(), "end": observed_time.isoformat()},
       "particles": int(ok.sum()), "drift_hours": DRIFT_HOURS,
       "notes": "PLACEHOLDER geography (NorKyst sample forcing). Single "
                "deterministic run - NOT an uncertainty estimate. bbox padded "
                f"{PAD} deg as a crude stand-in for ensemble spread."}
(FIXTURES / "drift_origin.json").write_text(json.dumps(out, indent=2))
print(json.dumps(out, indent=2))
print("wrote fixtures/drift_origin.json")

## 12. Visual check + save fixtures to Drive

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].imshow(np.array(Image.open(chosen).convert("L")), cmap="gray")
ax[0].set_title(f"SAR scene\n{chosen.name}")
ax[1].imshow(kept, cmap="gray", interpolation="nearest")
ax[1].set_title(f"detected + screened\n{int(kept.sum())} px")
step = max(1, dlon.shape[0] // 80)
for i in range(0, dlon.shape[0], step):
    ax[2].plot(dlon[i], dlat[i], lw=0.6, alpha=0.45, color="tab:blue")
ax[2].scatter(dlon[:, 0], dlat[:, 0], s=6, color="tab:red", label="observed slick")
ax[2].scatter(final_lon, final_lat, s=6, color="tab:green", label="backtracked origin")
ax[2].set_title(f"backward drift {DRIFT_HOURS}h\n(PLACEHOLDER geography)")
ax[2].legend(fontsize=8); ax[2].grid(alpha=.3)
for a in ax[:2]: a.axis("off")
plt.tight_layout(); plt.savefig("/content/pipeline_end_to_end.png", dpi=120); plt.show()

dest = Path("/content/drive/MyDrive/oil_spill_runs/fixtures"); dest.mkdir(parents=True, exist_ok=True)
for f in FIXTURES.iterdir():
    shutil.copy(f, dest / f.name)
shutil.copy("/content/pipeline_end_to_end.png", dest / "pipeline_end_to_end.png")
print("fixtures copied to Drive:", dest)
for f in sorted(dest.iterdir()):
    print("   ", f.name, f"({f.stat().st_size} bytes)")

## What this established, and what it did not

**Established:** the chain runs end to end. A real model prediction becomes
screened blobs, becomes lat/lon seeds, becomes a backward drift, becomes an
origin bbox and time window in the agreed contract format. Agent B can now score
against genuine model output.

**Not established:**
1. **Real geography.** Placeholder anchor inside the NorKyst domain. Needs CMEMS
   currents for Indian waters — a config change, but blocked on registration.
2. **Uncertainty.** One deterministic run with a fixed 0.02 deg pad. A defensible
   origin needs an ensemble over perturbed seed time, position and wind drift
   factor, reported as a probability field.
3. **Wind.** Currents only. Surface oil is strongly wind-driven; without it the
   origin estimate is biased.
4. **Validation.** No ground-truth spill origin to check against. This shows the
   pipeline is coherent, not that it is accurate.